# SpatialRoom3D: chạy riêng B0 (3DGS gốc của Inria) trên Runpod

Notebook này nằm trong repo (`app/notebooks/b0_inria.ipynb`). Nó chạy riêng **B0**: 3DGS gốc (Kerbl et al., SIGGRAPH 2023) bằng code chính thức của Inria (`graphdeco-inria/gaussian-splatting` @54c035f, env `gs-inria`), tham số như `full_eval.py`. Là mốc đối chiếu: `default` của gsplat chỉ được dùng làm baseline khi khớp B0 trên Deep Blending.

Đây là đúng phần B0 của thực nghiệm chính: Deep Blending (`drjohnson`, `playroom`), 30.000 vòng, cùng thư mục kết quả với code tổng (`runpod.ipynb`, ô 7, `EXP = "main"`; `scripts/pod/run_experiment.sh main --only b0`). Lượt đã xong ở đây được code tổng dùng lại, và ngược lại: không train hai lần.

**Tên bản: `b0_inria`.** Việc nền, báo cáo (`/workspace/outputs/reports/b0_inria_<thời gian>.md`), log và gói kết quả (`ket_qua_b0_inria_<thời gian>.tar.gz`) của notebook này đều mang tên đó; code tổng là `main`. Ghi chú, báo cáo đồ án: gọi đúng tên này.

**Cách dùng:** tải file notebook này lên `/workspace` (kéo thả vào thanh file bên trái), mở bản ở `/workspace`, chạy từng ô bằng **Shift+Enter**. Không mở thẳng bản trong repo: Jupyter ghi kết quả chạy vào file, làm repo trên pod có thay đổi và ô 3 sẽ không cập nhật code. Chép lại khi cần: `cp /workspace/SpatialRoom3D/app/notebooks/b0_inria.ipynb /workspace/`.

| Lúc nào | Chạy các ô |
|---|---|
| Lần đầu trên pod mới, volume trống: cài môi trường (cả 3 env, dùng chung với các notebook khác) | 1 → 2 → 3 → 4 (theo dõi ở ô 5) |
| Chạy riêng B0 trên benchmark, tự tải dữ liệu còn thiếu (tốn GPU khoảng 1 giờ) | 6 (`CHAY = True`), theo dõi ở ô 5 |
| Bảng và biểu đồ so sánh của lần chạy benchmark gần nhất | 7 |
| Xem mọi kết quả, đóng gói để tải về | 8 |
| Dừng mọi việc đang chạy | 9 (`DUNG=1`) |

- Ô 1–5 giống hệt `runpod.ipynb`: đã cài môi trường bằng notebook khác thì ô 4 chỉ kiểm tra lại.
- B0 chỉ chạy trên benchmark: repo chưa có luồng B0 cho cảnh tự chụp. B0 trên Mip-NeRF 360 (không thuộc thực nghiệm chính): `runpod.ipynb`, ô 7, `WHAT = "b0"`, `BENCH = "mipnerf360"`.
- Việc dài đều chạy nền: đóng trình duyệt không sao, nhưng **pod vẫn tính tiền** tới khi Stop / Terminate. Mỗi lúc chỉ một việc dùng GPU; việc gửi sau tự xếp hàng.
- Xong việc: tải gói `ket_qua_b0_inria_<thời gian>.tar.gz` (ô 8 in đường dẫn), rồi **Terminate pod**. Dữ liệu trên volume vẫn còn.

## 1. Kiểm tra máy

In [ ]:
%%bash
ver_ge() { [ "$(printf '%s\n%s\n' "$2" "$1" | sort -V | head -n 1)" = "$2" ]; }
gpu=$(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null | head -n 1)
cc=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader 2>/dev/null | head -n 1 | tr -d ' ')
vram=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits 2>/dev/null | head -n 1 | tr -d ' ')
drv=$(nvidia-smi 2>/dev/null | grep -oE 'CUDA Version: *[0-9]+\.[0-9]+' | grep -oE '[0-9]+\.[0-9]+' | head -n 1)
nv=$(nvcc --version 2>/dev/null | grep -oE 'release [0-9]+\.[0-9]+' | grep -oE '[0-9]+\.[0-9]+')
gccv=$(gcc -dumpversion 2>/dev/null)
fst=$(findmnt -n -o FSTYPE --target /workspace 2>/dev/null)
echo "GPU:         ${gpu:-KHÔNG THẤY GPU} (compute capability ${cc:-?})"
echo "Driver CUDA: ${drv:-?} | nvcc: ${nv:-không có} | gcc: ${gccv:-?} | /workspace: ${fst:-?}"
echo "----------------------------------------"
ok=1
if [ -z "$drv" ]; then echo "✗ Không thấy GPU hoặc driver: deploy lại pod"; ok=0
elif ver_ge "$drv" 12.8; then echo "✓ Driver hỗ trợ CUDA $drv: dùng PyTorch cu128"
elif ver_ge "$drv" 12.6; then echo "! Driver chỉ hỗ trợ CUDA $drv: ở ô 4 đặt TORCH_CUDA_OVERRIDE=\"cu126\""
else echo "✗ Driver CUDA $drv quá cũ: Terminate rồi deploy máy khác"; ok=0; fi
case "$cc" in
  8.0|8.6|8.9|9.0) echo "✓ GPU có trong danh sách kiến trúc build sẵn (8.0/8.6/8.9/9.0)" ;;
  *) echo "✗ GPU compute capability ${cc:-?} không được hỗ trợ: dùng RTX 4090, A6000, A40, L40S, A100 hoặc H100"; ok=0 ;;
esac
if [ "${nv%%.*}" = "12" ]; then echo "✓ nvcc $nv"; else echo "✗ nvcc ${nv:-không có}: sai template, cần image runpod/pytorch:2.4.0-py3.11-cuda12.4.1-devel-ubuntu22.04"; ok=0; fi
if [ -n "$gccv" ] && [ "${gccv%%.*}" -le 12 ]; then echo "✓ gcc $gccv"; else echo "✗ gcc ${gccv:-?}: cần gcc 12 trở xuống (template Ubuntu 22.04)"; ok=0; fi
if [ -n "$drv" ] && ! ver_ge "$drv" 12.9; then echo "! Driver CUDA $drv dưới 12.9: COLMAP dùng bản CPU (chậm hơn nhưng chạy được, tự chọn)"; fi
if [ "${vram:-0}" -ge 23000 ] 2>/dev/null; then echo "✓ VRAM ${vram} MiB"; else echo "! VRAM ${vram:-?} MiB dưới 24 GB: có thể thiếu bộ nhớ"; fi
case "$fst" in
  fuse*) echo "✓ /workspace là network volume" ;;
  *) echo "! /workspace là ổ '${fst:-?}', không phải network volume: Terminate pod sẽ mất dữ liệu" ;;
esac
if python3 -c 'import yaml' 2>/dev/null; then echo "✓ python3 có PyYAML (đọc configs/*.yaml)"
else echo "! python3 chưa có PyYAML: chạy 'python3 -m pip install pyyaml', hoặc chờ ô 4 cài xong env gs-gsplat (có sẵn PyYAML)"; fi
command -v git > /dev/null && command -v ssh-keygen > /dev/null && echo "✓ git, ssh có sẵn" || { echo "✗ thiếu git hoặc ssh"; ok=0; }
[ -d /workspace/miniforge3 ] && echo "! volume đã có /workspace/miniforge3 (không phải volume trống?)" || echo "✓ volume trống: ô 4 sẽ cài môi trường"
if [ "$ok" = 1 ]; then echo "=> ĐẠT: chạy tiếp ô 2"; else echo "=> CHƯA ĐẠT: sửa các dòng ✗ trước"; fi

## 2. Khóa deploy cho repo private (một lần cho mỗi volume)

Repo `SpatialRoom3D` là private. Pod đọc repo bằng một **deploy key chỉ đọc**: khóa chỉ mở được đúng repo này, không push được.

1. Chạy ô này: tạo khóa trong `/workspace/.ssh` (nếu chưa có) và in **khóa công khai**.
2. Trên GitHub: repo `SpatialRoom3D` → **Settings → Deploy keys → Add deploy key**, dán dòng khóa công khai, **không** tick "Allow write access".
3. Chạy lại ô này: dòng cuối phải là `✓ GitHub nhận khóa`.

Khóa riêng nằm trên volume (còn khi Terminate pod), không bao giờ được in hay gửi đi.

In [ ]:
%%bash
KEY=/workspace/.ssh/github_deploy
mkdir -p /workspace/.ssh && chmod 700 /workspace/.ssh
if [ -f "$KEY" ]; then
  echo "Khóa đã có: $KEY (không tạo lại)"
else
  ssh-keygen -t ed25519 -N "" -C "runpod-spatialroom3d" -f "$KEY" -q && echo "Đã tạo khóa mới: $KEY"
fi
chmod 600 "$KEY"
echo
echo "Khóa công khai, dán vào GitHub (SpatialRoom3D → Settings → Deploy keys → Add deploy key, KHÔNG tick 'Allow write access'):"
cat "$KEY.pub"
echo
SSH_CMD="ssh -i $KEY -o IdentitiesOnly=yes -o UserKnownHostsFile=/workspace/.ssh/known_hosts -o StrictHostKeyChecking=accept-new"
if $SSH_CMD -T git@github.com 2>&1 | grep -q "successfully authenticated"; then
  echo "✓ GitHub nhận khóa: chạy tiếp ô 3"
else
  echo "✗ GitHub chưa nhận khóa: thêm khóa công khai ở trên vào Deploy keys của repo SpatialRoom3D, rồi chạy lại ô này"
fi

## 3. Lấy code vào `/workspace/SpatialRoom3D`

- Chưa có: clone nhánh `BRANCH` từ GitHub bằng khóa của ô 2.
- Đã có: `git pull` (chỉ tiến lên, không ghi đè). Code trên pod có thay đổi chưa commit thì **không cập nhật** và in danh sách thay đổi; ô này không xóa hay cất gì.
- Không chạy khi đang có việc nền.

In [ ]:
%%bash
REPO_SSH="git@github.com:truong0809/SpatialRoom3D.git"
BRANCH="main"
CODE=/workspace/SpatialRoom3D
KEY=/workspace/.ssh/github_deploy
SSH_CMD="ssh -i $KEY -o IdentitiesOnly=yes -o UserKnownHostsFile=/workspace/.ssh/known_hosts -o StrictHostKeyChecking=accept-new"
if [ -f "$CODE/scripts/pod/job.sh" ] && bash "$CODE/scripts/pod/job.sh" busy; then
  echo "Đang có việc chạy nền: không đổi code lúc này (xem ô 5)"; exit 0
fi
if [ -d "$CODE/.git" ]; then
  changes="$(git -C "$CODE" status --porcelain)"
  if [ -n "$changes" ]; then
    echo "Code trên pod có thay đổi chưa commit: KHÔNG cập nhật (không xóa, không cất gì)."
    echo "$changes" | head -n 20
    echo "Xem: git -C $CODE status"
    exit 1
  fi
  GIT_SSH_COMMAND="$SSH_CMD" git -C "$CODE" pull --ff-only origin "$BRANCH" || { echo "Không cập nhật được (xem lỗi ở trên)"; exit 1; }
elif [ -e "$CODE" ]; then
  echo "$CODE đã có nhưng không phải repo Git: đổi tên hoặc xem lại, rồi chạy lại ô này"; exit 1
else
  GIT_SSH_COMMAND="$SSH_CMD" git clone -q -b "$BRANCH" "$REPO_SSH" "$CODE" \
    || { echo "Không clone được: chạy lại ô 2 (khóa deploy) và kiểm tra tên repo"; exit 1; }
  git -C "$CODE" config core.sshCommand "$SSH_CMD"
  echo "Đã clone $REPO_SSH (nhánh $BRANCH) vào $CODE"
fi
mkdir -p /workspace/data/custom_scenes /workspace/outputs
echo "Commit: $(git -C "$CODE" log --oneline -1)"

## 4. Cài môi trường (nền, không train)

`scripts/pod/run_all.sh`: kiểm tra máy → Miniforge + env `gs-inria` (3DGS gốc của Inria) → env `gs-tools` (COLMAP, ffmpeg) → env `gs-gsplat` (kèm smoke test) → gói log. Phiên bản ghim trong `configs/environment/`. **Chỉ cài môi trường, không train.**

Lần đầu trên volume trống mất khoảng 80 phút (đo trên RTX 4090 ngày 17/10/2026: `gs-inria` 29 phút, `gs-gsplat` 32 phút, `gs-tools` 18 phút); pod vẫn tính tiền trong lúc cài. Chạy lại khi env đã có thì chỉ kiểm tra lại. Ô này xong ngay, việc chạy tiếp ở nền (theo dõi ở ô 5).

In [ ]:
%%bash
TORCH_CUDA_OVERRIDE=""   # "cu126" nếu ô 1 báo driver chỉ hỗ trợ CUDA 12.6
MAX_JOBS=""              # "4" nếu log báo Killed khi build gsplat (hết RAM)
cd /workspace/SpatialRoom3D || { echo "Chưa có /workspace/SpatialRoom3D: chạy ô 3"; exit 0; }
[ -n "$TORCH_CUDA_OVERRIDE" ] && export TORCH_CUDA_OVERRIDE
[ -n "$MAX_JOBS" ] && export MAX_JOBS
bash scripts/pod/run_all.sh --background

## 5. Theo dõi

Cập nhật mỗi 15 giây: trạng thái các việc và cuối log của việc đang chạy; tự dừng khi không còn việc nào. Bấm nút ■ (Interrupt) để thôi theo dõi: việc nền vẫn chạy.

In [ ]:
import subprocess, time
from IPython.display import clear_output

JOB = "/workspace/SpatialRoom3D/scripts/pod/job.sh"

def job(*args):
    return subprocess.run(["bash", JOB, *args], capture_output=True, text=True, errors="replace")

while True:
    status = job("status").stdout
    running = [line.split()[0] for line in status.splitlines()
               if line and not line[0].isspace() and "ĐANG CHẠY" in line]
    tail = job("log", running[0], "20").stdout if running else ""
    clear_output(wait=True)
    print(time.strftime("%H:%M:%S"), "(cập nhật mỗi 15 giây; bấm ■ để thôi theo dõi, việc nền vẫn chạy)\n")
    print(status)
    if running:
        print(f"---------- cuối log của {running[0]}")
        print(tail)
    if job("busy").returncode != 0:
        print("=> Không còn việc nào chạy hoặc xếp hàng.")
        break
    time.sleep(15)

## 6. Chạy B0 trên benchmark

`scripts/pod/run_experiment.sh main --only b0`, trong việc nền `b0_inria`: kiểm tra môi trường (env `gs-inria`) → kiểm tra chỗ trống → tải Deep Blending nếu chưa có → B0 trên `drjohnson`, `playroom` → bảng B0 so với số liệu Inria công bố → gói kết quả nhẹ.

- `CHAY = False`: chỉ in kế hoạch, ước lượng và dữ liệu đã có (không tải, không train). `CHAY = True`: xếp hàng; ước lượng trên RTX 4090, chưa phải số đo: khoảng 1 giờ GPU (số đo ngày 17/10/2026 của repo trước: 49,5 phút, gồm cả tải dữ liệu), thêm khoảng 5 GB.
- Đặt `VOLUME_GB` đúng dung lượng volume: trước khi chạy, việc kiểm tra chỗ trống; không đủ thì dừng, lượt đã xong vẫn giữ.
- Luôn chạy đầy đủ (30.000 vòng). Lượt đã xong cùng cấu hình tự bỏ qua.

In [ ]:
CHAY = False              # True: xếp hàng việc nền (tốn GPU)
VOLUME_GB = 100           # dung lượng thật của network volume (GB)
AUTO_STOP = False         # True = dừng pod khi việc cuối cùng xong (chưa thử trên pod)

import os, subprocess
repo = "/workspace/SpatialRoom3D"
env = dict(os.environ, VOLUME_GB=str(VOLUME_GB), AUTO_STOP="1" if AUTO_STOP else "0")
if not os.path.isdir(repo):
    print(f"Chưa có {repo}: chạy ô 3.")
else:
    cmd = ["bash", f"{repo}/scripts/pod/run_experiment.sh", "main", "--only", "b0",
           "--background" if CHAY else "--plan"]
    out = subprocess.run(cmd, capture_output=True, text=True, errors="replace", cwd=repo, env=env)
    print(out.stdout[-12000:] + out.stderr[-3000:])
    if not CHAY:
        print("=> Chưa chạy: đây là kế hoạch. Đặt CHAY = True để xếp hàng (tốn GPU).")

## 7. Bảng và biểu đồ của lần chạy benchmark gần nhất (đọc file, không cần GPU)

In báo cáo mới nhất của ô 6 (`/workspace/outputs/reports/b0_inria_<thời gian>.md`, kẻ bảng sẵn): các bước và bảng B0 so với số liệu Inria công bố (PSNR / SSIM / LPIPS từng cảnh, trung bình, chênh lệch, kết luận); rồi các biểu đồ của lần chạy (ảnh trong `/workspace/outputs/reports/b0_inria_<thời gian>_charts/`, gộp các benchmark; cách đọc và bảng số ở `index.md` của thư mục đó; gói kết quả nhẹ có cả các ảnh này). Việc nền chưa xong thì báo cáo chưa có hoặc là của lần trước.

In [ ]:
import glob, os, re
def show_report(report):
    """Bảng của báo cáo `report` (.md) và biểu đồ của nó (ảnh trong <báo cáo>_charts/)."""
    try:
        from IPython import get_ipython
        from IPython.display import Image, Markdown, display
    except ImportError:
        get_ipython = None
    if get_ipython is None or get_ipython() is None:  # ngoài Jupyter: in chữ, chỉ ghi tên ảnh
        display = None
    text = open(report, encoding="utf-8").read()
    if text.startswith("# Thực nghiệm main "):  # docs/design/experiments.md mục 3.6, bước 5
        print("Nhắc: trước khi chọn mcmc_indoor (vòng ghép), báo cáo của main chỉ dùng để kiểm điều kiện: các bước OK, "
              "đối chiếu Inria (default so với B0 và số công bố), đủ seed, độ lệch chuẩn của default, mcmc trên room. "
              "Chênh lệch default / mcmc trên các cảnh ngoài room: đọc sau khi đã ghi quyết định.")
    if display is None:
        print(text)
    else:  # ảnh trong báo cáo dẫn bằng đường dẫn tương đối, không hiện được ở đây: hiện riêng bên dưới
        lines = [line for line in text.splitlines() if not line.startswith(("![", "**Biểu đồ**"))]
        display(Markdown("\n".join(lines)))
    folder = report[:-3] + "_charts"
    if not os.path.isdir(folder):
        print("Báo cáo này chưa có biểu đồ (cần matplotlib: env gs-gsplat có sẵn; CHARTS=0 thì không vẽ).")
        return
    print(f"===== Biểu đồ: {folder} (cách đọc và bảng số của từng biểu đồ: {folder}/index.md)")
    for path in sorted(glob.glob(f"{folder}/[0-9][0-9]_*.png")):
        if display is None:
            print(os.path.basename(path))
        else:
            display(Image(filename=path))

# chỉ báo cáo tổng của lần chạy riêng này (không lấy bảng từng benchmark, không lấy chế độ khác có tên bắt đầu như vậy)
reports = sorted(path for path in glob.glob("/workspace/outputs/reports/b0_inria_*.md")
                 if re.fullmatch(r"b0_inria_\d{8}_\d{6}\.md", os.path.basename(path)))
if reports:
    print(reports[-1])
    show_report(reports[-1])
else:
    print("Chưa có báo cáo: chạy ô 6 với CHAY = True, chờ việc nền xong (ô 5).")

## 8. Mọi kết quả (đọc file) và gói kết quả nhẹ `ket_qua_b0_inria_<thời gian>.tar.gz` để tải về (không có checkpoint, PLY hay video)

Gói mang tên bản của notebook này nhưng gồm mọi kết quả nhẹ trên volume (báo cáo, log, JSON của mọi lần chạy), không chỉ của bản này.

In [ ]:
!bash /workspace/SpatialRoom3D/scripts/pod/results.sh
!bash /workspace/SpatialRoom3D/scripts/pod/pack_results.sh --name b0_inria

## 9. Dừng mọi việc đang chạy

Đặt `DUNG=1` rồi chạy ô: dừng các việc chạy nền (kể cả việc đang xếp hàng) và mọi tiến trình con. Chạy lại ô tương ứng sau đó: phần đã xong tự bỏ qua, lượt train dở được chạy lại từ đầu.

In [ ]:
%%bash
DUNG=0   # 1: dừng mọi việc chạy nền
if [ "$DUNG" != 1 ]; then echo "Chưa dừng gì: đặt DUNG=1 rồi chạy lại ô này"; exit 0; fi
[ -f /workspace/SpatialRoom3D/scripts/pod/job.sh ] && bash /workspace/SpatialRoom3D/scripts/pod/job.sh stop --all
nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

## Khi gặp lỗi

| Thấy trong log / ô | Cách xử lý |
|---|---|
| `✗ GitHub chưa nhận khóa`, `Permission denied (publickey)` (ô 2, 3) | Khóa công khai chưa được thêm vào Deploy keys của **đúng** repo `SpatialRoom3D`: thêm rồi chạy lại ô 2 |
| `Code trên pod có thay đổi chưa commit` (ô 3) | Có file trong `/workspace/SpatialRoom3D` bị sửa (vd. mở notebook trong repo): xem `git -C /workspace/SpatialRoom3D status`, tự quyết định giữ hay bỏ |
| `no Python with PyYAML` | `python3 -m pip install pyyaml`, hoặc chờ ô 4 cài xong env gs-gsplat |
| `driver supports CUDA 12.6 < 12.8` | Ô 4: đặt `TORCH_CUDA_OVERRIDE="cu126"`, chạy lại ô 4 |
| `Killed` khi build gsplat | Ô 4: đặt `MAX_JOBS="4"`, chạy lại ô 4 |
| `compute capability ... is not in TORCH_CUDA_ARCH_LIST` | GPU không được hỗ trợ (vd. T4, RTX 50xx): deploy GPU khác |
| `another job is using the GPU` | Đang có việc khác chạy: xem ô 5; muốn dừng thì ô 9 |
| `MẤT` ở ô 5 | Pod tắt hoặc việc bị giết giữa chừng: chạy lại ô vừa dùng |
| `B0 (about ... GB more) would not fit in a ... GB volume` (ô 6) | Volume không đủ chỗ: tăng dung lượng network volume trên trang Runpod, đặt `VOLUME_GB` bằng dung lượng mới ở ô 6 rồi chạy lại (phần đã xong tự bỏ qua) |
| `SHA-256 of ... expected`, `is damaged` (tải dữ liệu) | File tải về hỏng hoặc khác bản gốc: file đó đã bị xóa, chạy lại ô 6 để tải lại |
| `already exist(s) ... but are incomplete` (tải dữ liệu) | Thư mục cảnh đã có nhưng thiếu phần (vd. tự đưa lên dở dang): xem lại hoặc dời thư mục đó đi, rồi chạy lại; bước tải không bao giờ sửa dữ liệu đã có |
| `Kết luận: **FAIL**` ở ô 7, `B0 does NOT match the Inria reference` (log) | B0 lệch số liệu Inria công bố quá ngưỡng (0,5 dB / 0,01 / 0,01): xem log của việc nền (ô 5) và `docs/ENVIRONMENT_AUDIT.md`; chưa dùng `default` làm baseline khi chưa rõ lý do |
| Lỗi khác | Chạy ô 8 và gửi gói `ket_qua_*.tar.gz` |